In [16]:
import os
import pandas as pd
import pmdarima as pm
from pmdarima import model_selection
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np
import warnings
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

In [17]:
def cast_to_time(df):
    df['time'] = pd.to_datetime(df['time'])
    return df

In [18]:
data_dir = '../Data/Preprocessed'
base_paths = os.listdir(data_dir)
full_paths = [os.path.join(data_dir, p) for p in base_paths]
patients = {base_paths[i].split('.')[0] : cast_to_time(pd.read_csv(fp, sep=';')) for i, fp in enumerate(full_paths)}
ids = list(patients.keys())

In [19]:
too_long = ["HUPA0027P", "HUPA0028P", "HUPA0026P"]
for i in too_long:
    ids.remove(i)
    s = 1.5

In [20]:
warnings.filterwarnings("ignore")
metrics = []
for i in ids:
    print(f"patient : {i}")
    ts = patients[i].loc[:, "glucose"]
    train_data = ts.iloc[:-220]
    test_data = ts.iloc[-220:]
    tscv = TimeSeriesSplit(n_splits=5, test_size=3)

    # Store the Mean Squared Errors (MSE) from each fold
    mse_scores = []
    mape_scores = []
    r_2_scores = []
    fold_results = []
    f = 1

    # Convert the pandas Series to a numpy array for splitting
    X = train_data.values

    # Iterate through the splits
    for train_index, test_index in tscv.split(X):
        # Split the data
        cv_train = X[train_index]
        cv_test = X[test_index]


        cv_model = pm.ARIMA(order=(4,0,3))
        cv_model.fit(cv_train)

        # --- Make Predictions on the Test Fold ---
        # The 'n_periods' is the length of the test set
        forecast, conf_int = cv_model.predict(n_periods=len(cv_test), return_conf_int=True)

        # --- Evaluate ---
        mse = mean_squared_error(cv_test, forecast)
        mape = mean_absolute_percentage_error(cv_test, forecast)
        r_2 = r2_score(cv_test, forecast)
        mse_scores.append(mse)
        mape_scores.append(mape)
        r_2_scores.append(r_2)
        # Store results for plotting
        fold_results.append({
            'train_data': train_data.iloc[train_index],
            'test_data': train_data.iloc[test_index],
            'forecast': pd.Series(forecast, index=train_data.index[test_index])
        })

        print(f"Fold: {f}")
        time.sleep(s/5)
        f += 1

    # Calculate and print the overall cross-validation score
    avg_mse = np.mean(mse_scores)
    avg_mape = np.mean(mape_scores)
    avg_r2 = np.mean(r_2_scores)
    metric = [avg_mse, avg_mape, avg_r2]
    print(f"\nAverage Cross-Validation MSE: {avg_mse:.2f}")
    print(f"\nAverage Cross-Validation MAPE: {avg_mape:.2f}")
    print(f"\nAverage Cross-Validation R2: {avg_r2:.2f}")
    
    metrics.append(metric)
    time.sleep(s/3)

patient : HUPA0018P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 4.89

Average Cross-Validation MAPE: 0.02

Average Cross-Validation R2: -2.85
patient : HUPA0014P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 9.40

Average Cross-Validation MAPE: 0.01

Average Cross-Validation R2: -10.34
patient : HUPA0019P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 11.15

Average Cross-Validation MAPE: 0.01

Average Cross-Validation R2: 0.64
patient : HUPA0007P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 243.93

Average Cross-Validation MAPE: 0.21

Average Cross-Validation R2: -10.59
patient : HUPA0003P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 2.91

Average Cross-Validation MAPE: 0.03

Average Cross-Validation R2: -25.36
patient : HUPA0022P
Fold: 1
Fold: 2
Fold: 3
Fold: 4
Fold: 5

Average Cross-Validation MSE: 3.66

Average Cross-Validation MAPE: 0.01

Average Cross-Va